# qec-bench — cloud training run

Trains the per-distance neural decoders on a free GPU (Colab T4 / Kaggle) for
configurations too large for a laptop CPU, and hands back the checkpoints the
benchmark harness scores.

**Runtime -> Change runtime type -> T4 GPU** before running.

**Resilient to disconnects.** The dataset stage checks Google Drive for a prior
result before doing any work and backs itself up as soon as it finishes; training
checkpoints are written *directly to Drive after every epoch*, so a dropped
session costs at most one epoch. Reopen the notebook, reconnect a GPU runtime,
and **Run all** — finished stages are skipped and training resumes from the last
completed epoch.

In [ ]:
!nvidia-smi -L

## Mount Drive (the resume point)

Everything that survives a disconnect lives under
`/content/drive/MyDrive/qecbench_backup/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
BACKUP = "/content/drive/MyDrive/qecbench_backup"
os.makedirs(BACKUP, exist_ok=True)
os.makedirs(f"{BACKUP}/weights", exist_ok=True)
print("backup dir:", BACKUP)

In [ ]:
import os
if not os.path.isdir("/content/qec-bench"):
    !git clone https://github.com/Lucas-Maingi/qec-bench.git /content/qec-bench
%cd /content/qec-bench
!pip install -q -e ".[train]"
os.makedirs("data", exist_ok=True)

## 1/3 — Training dataset

Restored from Drive if a previous session already generated it; otherwise
generated with Stim (CPU-bound, a few minutes) and backed up immediately.

In [ ]:
import os, shutil

if not os.path.isdir("data/train_v1") and os.path.isdir(f"{BACKUP}/train_v1"):
    print("restoring dataset from Drive")
    shutil.copytree(f"{BACKUP}/train_v1", "data/train_v1")

if not os.path.exists("data/train_v1/meta.json"):
    !qecbench generate --config configs/datasets/train_v1.yaml --out data
    shutil.copytree("data/train_v1", f"{BACKUP}/train_v1")
    print("dataset generated and backed up to Drive")
else:
    print("dataset already present, skipping generation")

## 2/3 — Train the per-distance decoders

Checkpoints and metrics land directly in the Drive backup directory, one write
per epoch. Finished distances are skipped in milliseconds; an interrupted
distance resumes from its last completed epoch. Point `--config` at a larger
training config to scale up.

In [ ]:
!qecbench train --config configs/train/mlp_v1.yaml --data data/train_v1 --out "$BACKUP/weights" --device cuda

## 3/3 — Package the deployable weights

Only the checkpoints and their metrics logs need to leave the session — a few
MB. The benchmark itself belongs on the serving CPU, not this VM:

```bash
unzip qecbench_weights.zip -d weights/
qecbench benchmark --dataset data/benchmark_v1     --decoders "pymatching,fusion_blossom,neural:weights"     --out results/benchmark_v1.json
```

The latency numbers that matter are the ones measured on the hardware you ship
inference on — rerun the benchmark locally and commit that results file.

In [ ]:
!cd "$BACKUP" && zip -j /content/qecbench_weights.zip weights/*
from google.colab import files
files.download("/content/qecbench_weights.zip")